# Meilenstein 1: Daten-Integration und Qualitätssicherung
## Projekt: Predictive Analytics im Radsport – Grand Tour Stage Ranking

### Einleitung
Dieses Notebook bildet den ersten Schritt der Datenverarbeitungspipeline. Ziel ist es, die separat erhobenen Datensätze der **Renn-Ergebnisse** (Grand Tour Stages) und der **Fahrer-Biometrie** (Physiologische Daten & Spezialisierungen) zu einem konsistenten Master-Datensatz zu verschmelzen. (Quellen anfügen?!)

Ein präzises Ranking-Modell erfordert nicht nur die historischen Platzierungen, sondern auch das physische Profil der Athleten (Größe, Gewicht, Alter) sowie deren spezifische Stärken (Climbing, Time Trial, etc.). In diesem Prozess identifizieren wir zudem Datenlücken und bereinigen den Datensatz von Redundanzen, um eine saubere Grundlage für das spätere Feature Engineering und das Training des **XGBRanker-Modells** zu schaffen.

### Arbeitsschritte in diesem Notebook:
1. **Daten-Import:** Laden der Rohdaten im JSONL-Format.
2. **Vorbereitung Data Merging**
3. **Daten-Merging und Daten in  Long Format bringen:** Verknüpfung der Tabellen via `rider_url` + Genestede Daten in Long Format bringen.
4. **Data Cleaning:** Entfernung technisch irrelevanter Merkmale (z.B. Bild-URLs).
5. **Explorative Lücken-Analyse:** Identifikation und Visualisierung fehlender Werte (`NaN`).
6. **Checkpointing:** Speicherung des bereinigten Zustands als persistente Pickle-Datei.

---

### 1. Daten Import:
Laden der Rohdaten im JSONL Format

In [18]:
# Laden der nötigen Packages

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [19]:
# 0. Test Directory
#print("Aktuelles Verzeichnis:", os.getcwd())

# 1. DATEN-IMPORT
# Pfade definieren (Passe diese an, falls deine Dateien in Unterordnern liegen)
results_file = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/V1_grand_tour_stage_results.jsonl'
riders_file = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/riders_full_biometrics_clean.jsonl'

df_results = pd.read_json(results_file, lines=True)
df_riders = pd.read_json(riders_file, lines=True)

print(df_results.head(5))
print(100*"=")
print(df_riders['url_name'].head(10))


             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

                                            metadata  \
0  {'departure': 'Fromentine', 'arrival': 'Noirmo...   
1  {'departure': 'Challans', 'arrival': 'Les Essa...   
2  {'departure': 'La Châtaigneraie', 'arrival': '...   
3  {'departure': 'Tours', 'arrival': 'Blois', 'di...   
4  {'departure': 'Chambord', 'arrival': 'Montargi...   

                                             results  
0  [{'rank': '1', 'rider_url': 'david-zabriskie',...  
1  [{'rank': '1', 'rider_url': 'tom-boonen', 'tim...  
2  [{'rank': '1', 'rider_url'

### 2. Vorbereitung Data Merging
- Explode (Zeilen-Vervielfältigung): Die Liste der Platzierungen in der Spalte stage_results wird "gesprengt", sodass aus einer Zeile pro Etappe ca. 150 bis 180 einzelne Zeilen werden – eine für jedes Fahrergebnis.
- Spalten Extraktion: Die in den Zeilen enthaltenen Dictionaries (mit Infos wie Platzierung, Zeit und Fahrer-URL) werden in flache, eigenständige Spalten umgewandelt, damit sie mathematisch auswertbar sind.
-Merge-Vorbereitung: Die extrahierte rider_url wird als eindeutiger Schlüssel (Key) genutzt, um im nächsten Schritt die körperlichen Merkmale aus der Biometrie-Tabelle passgenau an jedes einzelne Ergebnis anzuspielen.

In [20]:
df_results_long = df_results.explode('results').reset_index(drop=True) #Reset Index damit nicht alle Stage 1 Platzierungen gleichen Index haben
#print(df_results_long.head(5))
#Nun jeder Rank für jede Etappe eine Zeile

# Extraktion der Result Daten in einzelne Spalten
results_flat = pd.json_normalize(df_results_long['results'])
#results_flat.head(5)


# Zusammenführen der df_results_long (ohne results) mit results_flat
df_results_final = pd.concat([df_results_long.drop(columns=['results']),
                              results_flat], axis = 1)

df_results_final.head(10)

,race,year,url,metadata,rank,rider_url,time_gap
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",1,david-zabriskie,20:5120:51
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",2,lance-armstrong,0:020:02
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",3,alexandr-vinokourov,0:530:53
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",4,george-hincapie,0:570:57
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",5,laszlo-bodrogi,0:590:59
5,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",6,floyd-landis,1:021:02
6,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",7,fabian-cancellara,",,1:02"
7,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",8,jens-voigt,1:041:04
8,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",9,vladimir-karpets,1:051:05
9,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,"{'departure': 'Fromentine', 'arrival': 'Noirmo...",10,igor-gonzalez-de-galdeano,1:061:06


### 3. Daten-Merging und Daten in  Long Format bringen
- Tabellen-Merge (Left Join): Die Rennergebnisse werden über die rider_url mit der Biometrie-Tabelle verknüpft, sodass jedes Ergebnis um die Fahrer-Infos (Größe, Gewicht, etc.) ergänzt wird.


In [21]:
df_final = pd.merge(
    df_results_final,
    df_riders,
    left_on='rider_url',
    right_on='url_name',
    how='left'
)

print(df_final.head(5))

# Prüfung der fehlenden 88 Fahrerprofile
missing_rows = df_final['height'].isnull().sum()
print()

print(df_final.shape[0])
print(missing_rows)

# somit 32731 zeilen vorhanden
# 332 Zeilen nicht vorhanden -> Fahrerprofile fehlen


print(50*"=" + "\n")

# Ausgabe aller Spaltennamen

print(df_final.columns.to_list())

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

                                            metadata rank  \
0  {'departure': 'Fromentine', 'arrival': 'Noirmo...    1   
1  {'departure': 'Fromentine', 'arrival': 'Noirmo...    2   
2  {'departure': 'Fromentine', 'arrival': 'Noirmo...    3   
3  {'departure': 'Fromentine', 'arrival': 'Noirmo...    4   
4  {'departure': 'Fromentine', 'arrival': 'Noirmo...    5   

             rider_url    time_gap   birthdate  height  \
0      david-zabriskie  20:5120:51   1979-1-12    1.83   
1      lance-armstrong    0:020:02   1971-9-18  

- im nächsten Schritt die verbleibenden genesteden Daten in Long Format bringen

In [22]:
# metadata Spalte
df_meta = df_final['metadata'].apply(pd.Series)
#print(df_meta.head(10))

# zusammenführen mit dem Main Datensatz

df_final = pd.concat([df_final.drop(columns=['metadata']), df_meta], axis=1)
print(df_final.head(10))

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
5  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
6  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
7  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
8  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
9  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

  rank                  rider_url    time_gap   birthdate  height  \
0    1            david-zabriskie  20:5120:51   1979-1-12    1.83   
1  

In [23]:
#Spezialitätenprofil entpacken
df_spec = df_final['points_per_speciality'].apply(pd.Series).fillna(0)
df_final = pd.concat([df_final.drop(columns=['points_per_speciality']), df_spec], axis=1)
print(df_final.head(10))

             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
5  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
6  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
7  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
8  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
9  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

  rank                  rider_url    time_gap   birthdate  height  \
0    1            david-zabriskie  20:5120:51   1979-1-12    1.83   
1  

In [24]:

print(df_final['won_how'].unique())
print(df_final.columns.to_list())
df_final

['Time trial' 'Sprint of large group' '0.9 km solo' 'Sprint à deux'
 '76 km solo' '37 km solo' '? km solo' 'Sprint of small group' '1 km solo'
 '1.5 km solo' '-' '31 km solo' '8 km solo' '2 km solo' '70 km solo'
 '126 km solo' '0.4 km solo' '21 km solo' '18 km solo' '44 km solo'
 '15 km solo' '4 km solo' '30 km solo' '12.7 km solo' '4.7 km solo'
 '5.9 km solo' '5.3 km solo' '50 km solo' '11 km solo' '5.7 km solo'
 '2.1 km solo' '19 km solo' '13.5 km solo' '7.3 km solo' '29 km solo'
 '5.6 km solo' '2.2 km solo' '6.9 km solo' '18.9 km solo' '10.3 km solo'
 '10 km solo' '35 km solo' '1.7 km solo' '5 km solo' '1.4 km solo'
 '19.4 km solo' '1.9 km solo' '6.2 km solo' '25 km solo' '59.1 km solo'
 '0.8 km solo' '2.7 km solo' '3.3 km solo' '8.5 km solo' '4.6 km solo'
 '2.4 km solo' '12 km solo' '3.4 km solo' '0.7 km solo' '6.5 km solo'
 '48.4 km solo' '7.7 km solo' '0.5 km solo' '18.5 km solo' '48.7 km solo'
 '16.2 km solo' '6.4 km solo' '17.4 km solo' '26.8 km solo' '15.5 km solo'
 '12.8 km s

,race,year,url,rank,rider_url,time_gap,birthdate,height,image_url,name,nationality,place_of_birth,points_per_season_history,season_results,teams_history,weight,url_name,grand_tour_history,departure,arrival,distance,vertical_meters,profile_score,won_how,avg_speed,race_ranking,one_day_races,gc,time_trial,sprint,climber,hills,0
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1,david-zabriskie,20:5120:51,1979-1-12,1.83,None,David Zabriskie,US,Salt Lake City,"[{'season': 2013, 'points': 1, 'rank': 2540}, ...",[{'stage_url': 'race/il-lombardia/2013/result'...,"[{'season': 2013, 'team_name': 'Garmin Sharp',...",67.0,david-zabriskie,"[tour-de-france_2012, tour-de-france_2011, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,31.0,1075.0,5000.0,86.0,644.0,244.0,0.0
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2,lance-armstrong,0:020:02,1971-9-18,1.78,None,Lance Armstrong,US,Plano,"[{'season': 1998, 'points': 315, 'rank': 151},...","[{'stage_url': 'race/tour-down-under/2011/gc',...","[{'season': 2011, 'team_name': 'RadioShack', '...",72.0,lance-armstrong,"[tour-de-france_2010, tour-de-france_2009, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,3135.0,1410.0,8149.0,258.0,6159.0,1318.0,0.0
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3,alexandr-vinokourov,0:530:53,1973-9-16,1.77,None,Alexandr Vinokurov,KZ,Beskol,"[{'season': 2012, 'points': 496, 'rank': 105},...",[{'stage_url': 'race/san-sebastian/2012/result...,"[{'season': 2012, 'team_name': 'Astana Pro Tea...",68.0,alexandr-vinokourov,"[tour-de-france_2012, tour-de-france_2011, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,3038.0,5831.0,5198.0,356.0,5218.0,1322.0,0.0
3,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,4,george-hincapie,0:570:57,1973-6-29,1.91,None,George Hincapie,US,Queens,"[{'season': 2012, 'points': 80, 'rank': 673}, ...",[{'stage_url': 'national-race/el-tour-de-tucso...,"[{'season': 2012, 'team_name': 'BMC Racing Tea...",83.0,george-hincapie,"[tour-de-france_2012, tour-de-france_2011, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,3872.0,2018.0,3532.0,1722.0,1233.0,1374.0,0.0
4,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,5,laszlo-bodrogi,0:590:59,1976-12-11,1.87,images/riders/cw/ee/laszlo-bodrogi-2012.jpg,László Bodrogi,FR,Budapest,"[{'season': 2012, 'points': 135, 'rank': 453},...",[{'stage_url': 'race/chrono-des-nations/2012/r...,"[{'season': 2012, 'team_name': 'Team Type 1 - ...",79.0,laszlo-bodrogi,"[tour-de-france_2005, tour-de-france_2003, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,700.0,1025.0,5524.0,200.0,246.0,187.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225687,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,jan-hirt,",,0:00",1991-1-21,1.81,images/riders/gu/dq/jan-hirt-2026.jpeg,Jan Hirt,CZ,Třebíč,"[{'season': 2026, 'points': 5, 'rank': 1160}, ...",[{'stage_url': 'race/gran-premio-miguel-indura...,"[{'season': 2026, 'team_name': 'NSN Cycling Te...",62.0,jan-hirt,"[tour-de-france_2024, tour-de-france_2020, gir...",Alalpardo,Madrid,108 km,1678,92,-,-,11,129.0,1880.0,30.0,8.0,1646.0,128.0,0.0
225688,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,nadav-raisberg,",,0:00",2001-03-29,1.78,,Nadav Raisberg,IL,Dafna,"[{'season': 2026, 'points': 10, 'rank': 968}, ...",[],"[{'season': 2026, 'team': 'NSN Cycling Team'},...",67.0,nadav-raisberg,[],Alalpardo,Madrid,108 km,1678,92,-,-,11,60.0,75.0,50.0,4.0,6.0,77.0,0.0
225689,vuelta-a-espana,2025,https://www.procyclingstats.com/race/vuelta-a-...,NR,jake-stewart,",,0:00",1999-10-2,1.79,images/riders/gu/dq/jake-stewart-2026.jpeg,Jake Stewart,GB,Coventry,"[{'season': 2026, 'points': 13, 'rank': 862}

In [31]:
# 1. Laden der Datums-Datei
dates_file = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/grand_tour_stages_with_dates.jsonl'
df_dates = pd.read_json(dates_file, lines=True)

# 2. Join über 'url'
# Wir nutzen 'url', da diese Spalte in beiden DataFrames vorhanden ist.
# Wir holen uns 'stage_nr' und 'date' aus der neuen Datei dazu.
df_final_v2 = pd.merge(
    df_final,
    df_dates[['url', 'stage_nr', 'date']],
    on='url',
    how='left'
)

# 3. Datum in echtes datetime-Objekt umwandeln
df_final_v2['date'] = pd.to_datetime(df_final_v2['date'])

# 4. Kontrolle
print(f"Merge erfolgreich! Der Datensatz hat jetzt {df_final_v2.shape[0]} Zeilen.")
print(df_final_v2[['name', 'stage_nr', 'date']].head())
print(df_final_v2.head(10))

Merge erfolgreich! Der Datensatz hat jetzt 225692 Zeilen.
                 name  stage_nr       date
0     David Zabriskie         1 2005-07-02
1     Lance Armstrong         1 2005-07-02
2  Alexandr Vinokurov         1 2005-07-02
3     George Hincapie         1 2005-07-02
4      László Bodrogi         1 2005-07-02
             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
5  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
6  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
7  tour-de-france  2005  https://www.procyclingstats.com/rac

In [32]:
# Anzeigen komplette Daten
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
display(df_final_v2.head(3))

#Übersicht Na's pro Spalte
print("Anzahl der fehlenden Werte pro Spalte:")
print(df_final_v2.isnull().sum())


,race,year,url,rank,rider_url,time_gap,birthdate,height,image_url,name,nationality,place_of_birth,points_per_season_history,season_results,teams_history,weight,url_name,grand_tour_history,departure,arrival,distance,vertical_meters,profile_score,won_how,avg_speed,race_ranking,one_day_races,gc,time_trial,sprint,climber,hills,0,stage_nr,date
0,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,1,david-zabriskie,20:5120:51,1979-1-12,1.83,None,David Zabriskie,US,Salt Lake City,"[{'season': 2013, 'points': 1, 'rank': 2540}, ...",[{'stage_url': 'race/il-lombardia/2013/result'...,"[{'season': 2013, 'team_name': 'Garmin Sharp',...",67.0,david-zabriskie,"[tour-de-france_2012, tour-de-france_2011, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,31.0,1075.0,5000.0,86.0,644.0,244.0,0.0,1,2005-07-02
1,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,2,lance-armstrong,0:020:02,1971-9-18,1.78,None,Lance Armstrong,US,Plano,"[{'season': 1998, 'points': 315, 'rank': 151},...","[{'stage_url': 'race/tour-down-under/2011/gc',...","[{'season': 2011, 'team_name': 'RadioShack', '...",72.0,lance-armstrong,"[tour-de-france_2010, tour-de-france_2009, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,3135.0,1410.0,8149.0,258.0,6159.0,1318.0,0.0,1,2005-07-02
2,tour-de-france,2005,https://www.procyclingstats.com/race/tour-de-f...,3,alexandr-vinokourov,0:530:53,1973-9-16,1.77,None,Alexandr Vinokurov,KZ,Beskol,"[{'season': 2012, 'points': 496, 'rank': 105},...",[{'stage_url': 'race/san-sebastian/2012/result...,"[{'season': 2012, 'team_name': 'Astana Pro Tea...",68.0,alexandr-vinokourov,"[tour-de-france_2012, tour-de-france_2011, tou...",Fromentine,Noirmoutier-en-l'Ile,19 km,27,0,Time trial,54.676 km/h,n/a,3038.0,5831.0,5198.0,356.0,5218.0,1322.0,0.0,1,2005-07-02


Anzahl der fehlenden Werte pro Spalte:
race                             0
year                             0
url                              0
rank                             0
rider_url                        0
time_gap                         0
birthdate                     2874
height                        3932
image_url                    42746
name                          2874
nationality                   2874
place_of_birth                3131
points_per_season_history     2874
season_results                2874
teams_history                 2874
weight                        3932
url_name                      2874
grand_tour_history            2895
departure                        0
arrival                          0
distance                         0
vertical_meters                  0
profile_score                    0
won_how                          0
avg_speed                        0
race_ranking                     0
one_day_races                    0
gc              

In [ ]:
# Speichern des aktuellen Datenbestands in Pickle Format .pckl

output_dir = '/Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Den bereinigten DataFrame speichern
file_path = os.path.join(output_dir, '01_cleaned_master_data.pkl')
df_final_v2.to_pickle(file_path)

print(f"Checkpoint gesetzt! Datei gespeichert unter: {file_path}")
print(f"Datensatz enthält jetzt {df_final.shape[1]} Spalten und {df_final.shape[0]} Zeilen.")

Checkpoint gesetzt! Datei gespeichert unter: /Users/leonfrey/Documents/TU Dresden/10 Semester/Gestaltungsansätze der Advanced Data Analytics/2. Version/01_cleaned_master_data.pkl
Datensatz enthält jetzt 33 Spalten und 225692 Zeilen.


## **Weiter mit Explorations Notebook** (neu)

Daten dann folgendermaßen laden:
```
import pandas as pd
df = pd.read_pickle('../../data/processed/01_cleaned_master_data.pkl')
```

### Offene Spalten zum extrahieren:
- points per season -> Mit Etappenjahr mappen
- Team History -> mit Etappenjahr mappen

### Weitere To Do's
- Einheiten rausnehmen aus Spalten (Dokumentieren!)


- Ausprägungen in den Spalten analysieren (Zahlen, Diagramme)
- NAs prüfen + Umgang
- Säubern der Spalten die wir nicht brauchen
- Doku schreiben Spaltennamen
- Wetter API 
    - Koordinaten (über Start und Zielort)
    - Best Practise Umsetzung Datengestalt/ -Umfang